# Depth Anything Training Notebook

This notebook trains the prediction head of either **Depth-Anything V2** or **Depth Anything 3** on the same validation split.

It supports two modes:

1. `INITIAL_WEIGHTS' = "pretrained"`: finetunes the DPT/DualDPT head starting from the pretrained weights.
2. `INITIAL_WEIGHTS' = "reset"`: resets the DPT/DualDPT head weights.

# Setup

In [ ]:
# Optional install if packages are missing in the current environment
# %pip install torch numpy

# required for DA2 training
# %pip install transformers

# required for DA3 training
# %pip install xformers torchvision
# %pip install git+https://github.com/ByteDance-Seed/Depth-Anything-3.git

In [ ]:
from pathlib import Path 
from torch.utils.data import DataLoader, Subset
from dataset import TrainDataset, make_or_load_split
import numpy as np 
import json 
import torch
import torch.nn.functional as F 
import os

## Configuration

Change `MODEL_FAMILY` and `WEIGHTS_SOURCE` here to switch between 
`DA2`/`DA3` and starting training from pretrained/fresh weights.

The training data to evaluate the model on should be in the *data/train* directory and must consist of RGB images, each paired with a depth map saved as a numpy array

In [ ]:
# ========================================
# Configuration cell: edit only this cell 
# ========================================

# Choose the model family: "DA2" or "DA3"
MODEL_FAMILY = "DA3"

# Choose which weights to start with.
# "pretrained"  -> start training from pretrained weights
# "reset"  -> start training from random weights
WEIGHTS_SOURCE = "pretrained"

# Train/Validation split
VAL_FRAC = 0.10
SPLIT_SEED = 42

# Data and runtime settings.
PROJECT_ROOT = Path("..")
TRAIN_DIR = PROJECT_ROOT / "data/train"
SPLIT_DIR = PROJECT_ROOT / "splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
SPLIT_PATH = SPLIT_DIR / f"train_val_split_seed{SPLIT_SEED}_val{int(VAL_FRAC * 100)}.json"
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 1
NUM_WORKERS = 4
NUM_EPOCHS = 10

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Model

This section loads the selected pretrained Depth Anything model, freezes the full model, and then re-enables gradients only for the depth prediction head.

In [ ]:
def reset_weights(m):
    ''' 
    Helper function used when WEIGHTS_SOURCE == "reset".
    It calls reset_parameters() on modules that support it, such as Linear and Conv layers.
    '''
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()
        
if MODEL_FAMILY=="DA2":
    from transformers import AutoModelForDepthEstimation
    model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf")
    model = model.to(device)

    # Freeze all model parameters first so that the backbone does not change during training.
    for p in model.parameters():
        p.requires_grad = False 

    # Optionally reset the prediction head.
    if WEIGHTS_SOURCE=="reset":
        model.head.apply(reset_weights)
    elif WEIGHTS_SOURCE!="pretrained":
        raise ValueError('Invalid WEIGHTS_SOURCE. Accepted values are "reset" or "pretrained".')
    
    # Re-enable gradients only for the prediction head.
    for p in model.head.parameters():
        p.requires_grad = True 

    optimizer = torch.optim.AdamW(model.head.parameters(), lr=5e-5)

elif MODEL_FAMILY=="DA3":
    # DA3 wraps the actual PyTorch model inside model.model,
    # so the head is accessed through model.model.head.
    from depth_anything_3.api import DepthAnything3 
    model = DepthAnything3.from_pretrained('depth-anything/da3metric-large')
    model = model.to(device)

    # Freeze all model parameters first so that the backbone does not change during training.
    for p in model.parameters():
        p.requires_grad = False

    # Re-enable gradients only for the prediction head.
    for p in model.model.head.parameters():
        p.requires_grad = True 

    # Optionally reset the prediction head.      
    if WEIGHTS_SOURCE=="reset":
        model.model.head.apply(reset_weights)
    elif WEIGHTS_SOURCE!="pretrained":
        raise ValueError('Invalid WEIGHTS_SOURCE. Accepted values are "reset" or "pretrained".')

    optimizer = torch.optim.AdamW(model.model.head.parameters(), lr=5e-5)

else:
    raise ValueError('Invalid MODEL_FAMILY. Accepted values are "DA2" or "DA3".')

## Load Data

This section loads the training dataset, creates or reloads a deterministic train/validation split, and builds separate DataLoaders for training and validation.

In [ ]:
train_dataset = TrainDataset(str(TRAIN_DIR))

train_indices, val_indices = make_or_load_split(
    train_dataset=train_dataset,
    split_path=SPLIT_PATH,
    split_seed=SPLIT_SEED,
    val_frac=VAL_FRAC,
    train_dir=TRAIN_DIR
)

train_set = Subset(train_dataset, train_indices)
val_set = Subset(train_dataset, val_indices)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Training

### Training Loss Function

$$
\mathcal{L} = \mathcal{L}_D(\tilde{\mathbf{D}}, \mathbf{D}) + \mathcal{L}_{\text{grad}}(\tilde{\mathbf{D}}, \mathbf{D}) ,
$$

where $\tilde{\mathbf{D}}$ denotes the predicted depth map and $\mathbf{D}$ denotes the ground-truth depth map. The first term is the depth reconstruction loss, i.e., the mean absolute error (MAE):
$$
\mathcal{L}_D(\tilde{\mathbf{D}}, \mathbf{D})
=
\frac{1}{|\Omega|}
\sum_{p \in \Omega}
\left|
\tilde{\mathbf{D}}_p - \mathbf{D}_p
\right| ,
$$
where $\Omega$ is the set of valid depth pixels (at least $1mm$ and at most $80m$) and $p$ indexes individual pixels, and the second term is the gradient matching loss that penalizes differences between the local depth gradients of the prediction and the ground truth:
$$
\mathcal{L}_{\text{grad}}(\tilde{\mathbf{D}}, \mathbf{D})
=
\left\|
\nabla_x \tilde{\mathbf{D}} - \nabla_x \mathbf{D}
\right\|_1
+
\left\|
\nabla_y \tilde{\mathbf{D}} - \nabla_y \mathbf{D}
\right\|_1 ,
$$
where $\nabla_x$ and $\nabla_y$ denote horizontal and vertical finite-difference operators, respectively. This term encourages the predicted depth map to preserve sharp depth discontinuities and local geometric structure.

In [ ]:
def depth_loss(pred, target):
    # Build a validity mask so that invalid depth values do not contribute to the loss.
    mask = torch.isfinite(pred) & torch.isfinite(target) & (target >= 1e-3) & (target <= 80.0)

    # L1 depth penalizes absolute differences between predicted and target depth.
    l1 = F.l1_loss(pred[mask], target[mask])

    # Compute horizontal depth gradients for prediction and target.
    pred_dx = pred[..., :, 1:] - pred[..., :, :-1]
    target_dx = target[..., :, 1:] - target[..., :, :-1]

    # Compute vertical depth gradients for prediction and target.
    pred_dy = pred[..., 1:, :] - pred[..., :-1, :]
    target_dy = target[..., 1:, :] - target[..., :-1, :]

    # A gradient is valid only if both neighboring pixels are valid.
    mask_x = mask[..., :, 1:] & mask[..., :, :-1]
    mask_y = mask[..., 1:, :] & mask[..., :-1, :]

    loss_grad_x = F.l1_loss(pred_dx[mask_x], target_dx[mask_x])
    loss_grad_y = F.l1_loss(pred_dy[mask_y], target_dy[mask_y])

    # The final loss is the sum of the depth reconstruction loss and the gradient consistency loss.
    return l1 + (loss_grad_x + loss_grad_y)

### Scale Invariant RMSE

$$
\text{si-RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (\delta_i + \alpha)^2},
$$

where the $\log$ difference $\delta_i$ between the $i$-th pixel of prediction $\hat d$ and the ground truth $d$ is given by

$$\delta_i = \log(\hat d_i) - \log(d_i),$$

and the bias term $\alpha$ by

$$
\alpha = \frac{1}{n} \sum_{i=1}^n \left(\log(d_i) - \log(\hat d_i) \right). 
$$

In [ ]:
def si_rmse(pred, target):
    # Use the same validity mask as in the training loss.
    mask = torch.isfinite(pred) & torch.isfinite(target) & (pred > 1e-12) & (target >= 1e-3) & (target <= 80.0)

    # Clamp depths before taking the logarithm to avoid log(0) just for safety.
    pred_masked = pred[mask].clamp_min(1e-12)
    target_masked = target[mask].clamp_min(1e-12)

    # Compute log-depth values for prediction and target.
    log_pred = torch.log(pred_masked)
    log_target = torch.log(target_masked)

    # delta is the per-pixel log-depth error.
    delta = log_pred - log_target 

    # alpha removes the mean log-depth bias.
    alpha = torch.mean(log_target - log_pred)

    # Return the root mean squared log error after scale correction.
    return torch.sqrt(torch.mean((delta + alpha) ** 2))

### Single Epoch Training and Validation Routine

This section defines one training pass over the training set and one validation pass over the validation set.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    # Train only the head while keeping the backbone in evaluation mode.
    if MODEL_FAMILY == "DA2":
        model.eval()
        model.head.train()
    else:
        model.model.eval()
        model.model.head.train()

    total_loss = 0.0
    num_batches = 0

    for step, batch in enumerate(dataloader):
        images = batch["image"].to(device)
        target = batch["depth"].to(device)

        # Clear gradients from the previous optimization step.
        optimizer.zero_grad()

        if MODEL_FAMILY == "DA2":
            pred = model(images).predicted_depth.unsqueeze(1)
        else:
            # DA3 expects an additional dimension here, so the image tensor is unsqueezed before inference.
            images = images.unsqueeze(1)
            pred = model.model(images).depth

        # Compute training loss
        loss = depth_loss(pred, target)

        # Backpropagate through the trainable head parameters.
        loss.backward()
        
        # Update only the parameters passed to the optimizer.
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if step % 20 == 0:
            print(f"step {step:04d} | loss {loss.item():.4f}")

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def validate(model, dataloader, device):
    # Put the full model into evaluation mode during validation.
    if MODEL_FAMILY=="DA2":
        model.eval()
    else:
        model.model.eval()

    total_loss = 0.0
    num_batches = 0
    all_deltas = []

    for batch in dataloader:
        images = batch["image"].to(device)
        target = batch["depth"].to(device)

        # Run inference using the correct model API for the selected model family.
        if MODEL_FAMILY=="DA2":
            pred = model(images).predicted_depth.unsqueeze(1)
        else:
            images = images.unsqueeze(1)
            pred = model.model(images).depth

        loss = depth_loss(pred, target)

        total_loss += loss.item()
        num_batches += 1

        # Build a valid-pixel mask for the scale-invariant RMSE calculation.
        mask = (
            torch.isfinite(pred)
            & torch.isfinite(target)
            & (pred > 1e-12)
            & (target > 1e-3)
            & (target < 80.0)
        )

        # Compute log-depth errors only over valid pixels.
        log_pred = torch.log(pred[mask].clamp_min(1e-12))
        log_target = torch.log(target[mask].clamp_min(1e-12))

        delta = log_pred - log_target

        # Store detached CPU tensors so that validation does not keep GPU computation graphs in memory.
        all_deltas.append(delta.detach().cpu())

    # Average the validation loss across batches.
    avg_loss = total_loss / max(num_batches, 1)

    # Concatenate all per-pixel log-depth errors across the validation set.
    all_deltas = torch.cat(all_deltas, dim=0)

    # Compute the global scale correction alpha.
    alpha = torch.mean(-all_deltas)

    # Compute validation si-RMSE over all valid validation pixels.
    val_si_rmse = torch.sqrt(torch.mean((all_deltas + alpha) ** 2))

    return avg_loss, val_si_rmse.item()


### Actual Training Loop

This cell runs the full training process, records the metric history, saves a checkpoint after every epoch, and keeps a separate copy of the best checkpoint according to validation si-RMSE.

In [ ]:
best_val_si_rmse = float("inf")
history = {"train_loss": [], "val_loss": [], "val_si_rmse": [],}

# Run one full training and validation cycle per epoch.
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        device=device,
    )

    val_loss, val_si_rmse = validate(
        model=model,
        dataloader=val_loader,
        device=device,
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_si_rmse"].append(val_si_rmse)

    print(f"train loss:    {train_loss:.4f}")
    print(f"val loss:      {val_loss:.4f}")
    print(f"val si-RMSE:   {val_si_rmse:.4f}")

    # Extract only the trainable head weights.
    if MODEL_FAMILY == 'DA2':
        model_head = model.head.state_dict()
    else:
        model_head = model.model.head.state_dict()

    # Store enough information to resume or analyze training later.
    checkpoint = {
        "epoch": epoch + 1,
        "model_head": model_head,
        "optimizer": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_si_rmse": val_si_rmse,
        "history": history,
    }

    # Save an epoch-specific checkpoint after every epoch.
    torch.save(
        checkpoint,
        CHECKPOINT_DIR / f"{MODEL_FAMILY}_{WEIGHTS_SOURCE}_epoch_{epoch + 1:03d}.pt",
    )

    # If this is the best validation si-RMSE so far, also save as best checkpoint.
    if val_si_rmse < best_val_si_rmse:
        best_val_si_rmse = val_si_rmse

        torch.save(
            checkpoint,
            CHECKPOINT_DIR / f"{MODEL_FAMILY}_{WEIGHTS_SOURCE}_best.pt",
        )

        print("saved new best checkpoint")